In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import re
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)  # Show all columns in DataFrame display
import sys; sys.path.append('..')
from config import Paths
from eals_targetals.utils import make_article_table
from dataframes import *

In [3]:
df_demo = load_demographics_data()

### RAPA pALS vs Radcliff pALS

In [4]:
import sys; sys.path.append('/home/ubuntu/eals/eals_radcliff_feli/')
from eals_radcliff.utils import dataframes

In [5]:
# Load radcliff demographics data
df_rad_demo = dataframes.load_demographics_data()

# add year_of_diagnosis and symptom_onset_year
df_rad_demo['symptom_onset_year'] = df_rad_demo.symptom_onset_date.dt.year
s = df_rad_demo['date_of_diagnosis'].astype(str).str.strip()
s = s.str.replace(r'(?i)^O(?=\d)', '0', regex=True) # Replace leading/embedded 'O' that should be a zero ->  O7/15/2023 -> 07/15/2023
s = s.str.replace(r'(?i)(?<=\d)O(?=\d)', '0', regex=True) # 1O/15/2023 -> 10/15/2023 (if it happens)
df_rad_demo['year_of_diagnosis'] = pd.to_datetime(s, format='%m/%d/%Y', errors='coerce').dt.year

# Load radcliff ALSFRSR data to get age at enrollment
df_rad_als = dataframes.load_alsfrsr_data()
df_rad_als.date = pd.to_datetime(df_rad_als.date)
df_rad_als.sort_values(['user_id','date'], inplace=True)
df_rad_als = df_rad_als.groupby('user_id').first().reset_index()

# Add age at enrollment
age_at_enrollment = []
for idx, row in df_rad_demo.iterrows():
    pal_als = df_rad_als.query('user_id == @row.user_id')
    if pal_als.shape[0] == 0:
        r = np.nan
    else:
        r = pal_als.date.iloc[0].year - row['date_of_birth'].year
    age_at_enrollment.append(r)
df_rad_demo["age_at_enrollment"] = age_at_enrollment

In [6]:
# Merge Radcliff and RAPA data
df_rad = pd.merge(df_rad_demo, df_rad_als, on='user_id', how='outer')
df_rad['study'] = 'Radcliff'
print(df_rad.shape)

df_rapa_als = load_alsfrsr_data()
df_rapa_als.date = pd.to_datetime(df_rapa_als.date)
df_rapa_als.sort_values(['user_id','date'], inplace=True)
df_rapa_als = df_rapa_als.groupby('user_id').first().reset_index()
df_rapa = df_demo.merge(df_rapa_als, on='user_id', how='outer')
df_rapa['study'] = 'RAPA'
print(df_rapa.shape)

df_comparison = pd.concat([df_rapa, df_rad], axis=0, ignore_index=True, sort=False)
print(df_comparison.shape)

(156, 97)
(15, 98)
(171, 105)


In [7]:
# Example usage
cat_columns = [
    'birth_gender',
    'symptom_onset_site',
    'specific_limb_onset', 
    'family_neuro_history',
    'muscle_disease',
    'has_fallen',
    'any_walking_aid',
    'anxiety', 
    'depression',
]
numeric_columns = [
    'age',
    'height_in_m',
    'bmi',
    'years_since_onset',
    'ALS_total',
    'speech',
    'salvation',
    'swallowing',
    'handwriting',
    'cutting_food_a',
    'cutting_food_b',
    'dressing_and_hygiene',
    'turning_in_bed',
    'walking',
    'climbing_stairs',
    'dyspnea',
    'orthopnea',
    'respiratory_insufficiency',
    'bulbar_subscore',
    'respiratory_subscore',
    'fine_motor_subscore',
    'gross_motor_subscore',

]

In [8]:
styled_table = make_article_table(df_comparison, 'study', [], numeric_columns, dec=2, p_dec=2)
display(styled_table)

,,Radcliff,RAPA,p-value (T-Test/Chi-squared)
Variable,Value,,,
Age,Average (std) [N],59.59 (11.24) [141],58.00 (13.61) [15],0.67
Height In M,Average (std) [N],1.72 (0.09) [142],1.71 (0.10) [15],0.52
Bmi,Average (std) [N],27.24 (4.84) [142],24.30 (4.61) [15],0.031
Years Since Onset,Average (std) [N],4.25 (4.70) [132],3.76 (2.74) [14],0.56
Als Total,Average (std) [N],40.32 (5.53) [143],25.36 (9.22) [11],0.00029
Speech,Average (std) [N],3.50 (0.63) [143],2.18 (1.72) [11],0.03
Salvation,Average (std) [N],3.57 (0.74) [143],3.09 (0.94) [11],0.13
Swallowing,Average (std) [N],3.62 (0.57) [143],3.00 (1.18) [11],0.11
Handwriting,Average (std) [N],3.43 (0.60) [143],1.73 (1.68) [11],0.0073


In [9]:
styled_table = make_article_table(df_comparison, 'study', cat_columns, [], dec=2, p_dec=2)
display(styled_table)

Exact FFH not available. Falling back to chi-square p-value. Install 'fisher' for exact.
